In [ ]:
import os
import json
from collections import defaultdict
from tqdm import tqdm
import numpy as np

def load_json_file(file_path):
    """
    Load a JSON file.
    
    Args:
        file_path: Path to the JSON file
        
    Returns:
        Loaded JSON data
    """
    with open(file_path, 'r') as f:
        return json.load(f)

def union_json_files_with_top_k(json_files, codebase_dir, top_k):
    """
    Generate union of top-k paths from multiple JSON files and filter hallucinated paths.
    
    Args:
        json_files: List of paths to JSON files
        codebase_dir: Base directory containing codebases
        top_k: Only consider top-k predictions from each model
        
    Returns:
        Dictionary with union of filtered predictions
    """
    # Dictionary to store the union of paths for each instance_id
    union_predictions = defaultdict(set)
    
    # Load and process each JSON file
    print(f"Processing {len(json_files)} JSON files with top-{top_k}...")
    for file_path in tqdm(json_files):
        # Load JSON data
        json_data = load_json_file(file_path)
        
        # Add top-k paths to union
        for instance_id, paths in json_data.items():
            # Extract paths based on data structure
            if isinstance(paths, list):
                model_paths = paths[:top_k]  # Only take top-k paths
            elif isinstance(paths, dict) and "paths" in paths:
                model_paths = paths["paths"][:top_k]  # Only take top-k paths
            else:
                continue  # Skip invalid data
                
            # Add model's top-k paths to the union set
            for path in model_paths:
                union_predictions[instance_id].add(path)
    
    # Convert sets to lists for JSON serialization
    unfiltered_predictions = {instance_id: list(paths) for instance_id, paths in union_predictions.items()}
    
    # Filter out hallucinated paths
    filtered_predictions, hallucination_count, total_paths = filter_hallucinated_paths(
        unfiltered_predictions, codebase_dir
    )
    
    # Print hallucination statistics
    if total_paths > 0:
        hallucination_rate = (hallucination_count / total_paths) * 100
        print(f"Filtered {hallucination_count} hallucinated paths out of {total_paths} total paths ({hallucination_rate:.2f}%)")
    
    return filtered_predictions

def filter_hallucinated_paths(predictions, codebase_dir="./codebases"):
    """
    Filter out hallucinated paths from predictions.
    
    Args:
        predictions: Dictionary mapping instance_id to list of predicted paths
        codebase_dir: Base directory containing codebases
        
    Returns:
        Tuple of (filtered_predictions, hallucination_count, total_paths)
    """
    filtered_predictions = {}
    hallucination_count = 0
    total_paths = 0
    
    for instance_id, paths in predictions.items():
        filtered_paths = []
        for path in paths:
            total_paths += 1
            full_path = os.path.join(codebase_dir, instance_id, path)
            if os.path.exists(full_path):
                filtered_paths.append(path)
            else:
                hallucination_count += 1
        
        filtered_predictions[instance_id] = filtered_paths
    
    return filtered_predictions, hallucination_count, total_paths

def evaluate_predictions(predictions, ground_truth):
    """
    Evaluate prediction accuracy.
    
    Args:
        predictions: Dictionary mapping instance_id to list of predicted paths
        ground_truth: Dictionary mapping instance_id to ground truth path
        
    Returns:
        Dictionary with evaluation results
    """
    # Find common instance IDs
    common_ids = set(predictions.keys()) & set(ground_truth.keys())
    
    if not common_ids:
        print("No common instance IDs found between predictions and ground truth!")
        return {}
    
    correct_count = 0
    path_counts = []
    zero_path_count = 0
    
    for instance_id in common_ids:
        true_path = ground_truth[instance_id]
        pred_paths = predictions[instance_id]
        
        path_counts.append(len(pred_paths))
        
        if not pred_paths:
            zero_path_count += 1
        
        if true_path in pred_paths:
            correct_count += 1
    
    # Calculate results
    accuracy = correct_count / len(common_ids) if common_ids else 0
    avg_paths = np.mean(path_counts) if path_counts else 0
    
    return {
        "accuracy": float(accuracy),
        "avg_paths": float(avg_paths),
        "zero_path_count": zero_path_count,
        "total_instances": len(common_ids)
    }

# Hard-coded configuration
config = {
    "json_files": [
        "./localization/hierarchy_0/chatgpt/20250327_165440/predictions.json",
        "./localization/hierarchy_0/claude/20250327_165619/predictions.json",
        "./localization/hierarchy_0/deepseek/20250327_140625/predictions.json", 
        "./localization/hierarchy_0/grok/20250327_145800/predictions.json", 
        "./localization/hierarchy_0/mistral/20250327_114125/predictions.json"
    ],
    "codebase_dir": "./codebases",
    "ground_truth_path": "./ground_truth/bug_paths.json",
    "output_dir": "./localization_combination_results/union/selected_results",
    "top_k_values": [1, 2, 3, 5, 7, 10, 12, 15]
}

# Get configuration
json_files = config["json_files"]
codebase_dir = config["codebase_dir"]
ground_truth_path = config["ground_truth_path"]
output_dir = config["output_dir"]
top_k_values = config["top_k_values"]

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Load ground truth
print("Loading ground truth data...")
ground_truth = load_json_file(ground_truth_path)

# Results dictionary to store evaluation results for each top-k
all_results = {}

# Process each top-k value
for top_k in top_k_values:
    print(f"\nProcessing top-{top_k}...")
    
    # Generate union of JSON files with current top-k
    union_predictions = union_json_files_with_top_k(json_files, codebase_dir, top_k)
    
    # Save union results to output file
    output_file = os.path.join(output_dir, f"union_top_{top_k}_results.json")
    with open(output_file, 'w') as f:
        json.dump(union_predictions, f, indent=2)
    
    print(f"Saved union top-{top_k} results to {output_file}")
    print(f"Total unique instance IDs: {len(union_predictions)}")
    
    # Evaluate predictions
    evaluation_result = evaluate_predictions(union_predictions, ground_truth)
    all_results[top_k] = evaluation_result
    
    # Print current evaluation result
    print(f"Top-{top_k} Accuracy: {evaluation_result['accuracy']:.4f}")
    print(f"Top-{top_k} Avg Paths: {evaluation_result['avg_paths']:.2f}")
    print(f"Top-{top_k} Zero Path Count: {evaluation_result['zero_path_count']}")

# Save all evaluation results
eval_output_file = os.path.join(output_dir, "all_evaluation_results.json")
with open(eval_output_file, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\nSaved all evaluation results to {eval_output_file}")

# Print evaluation summary
print("\nEvaluation Summary:")
print("=" * 70)
print(f"{'Top-k':^6} | {'Accuracy':^10} | {'Avg Paths':^12} | {'Zero Path Count':^16} | {'Total Instances':^15}")
print("=" * 70)

for k in sorted(all_results.keys()):
    result = all_results[k]
    print(f"{k:^6} | {result['accuracy']:.4f}    | {result['avg_paths']:^12.2f} | {result['zero_path_count']:^16} | {result['total_instances']:^15}")